# Push database_files/ to Supabase

Reads credentials from `.env.local`, then pushes the 8 synced CSVs to their live Supabase tables.

- `prospects` is pushed first (every other table has a foreign key referencing it).
- `prospect_profiles`, `prospect_sources`, `prospect_scores`, `prospect_analysis`, `prospect_pipeline`: upsert on `prospect_id`.
- `prospect_sectors`: mirrored — deletes any sector row for a prospect that's no longer in the local list, then upserts the current set (unique constraint on `prospect_id, sector`).
- `prospect_diligence`: delete-then-insert per prospect (its primary key is an internal auto-increment `id` not present in our CSV, so a true upsert isn't possible).
- Several columns are `NOT NULL` with no usable blank representation — those get schema-appropriate defaults filled in *only for the outgoing payload*; the local CSVs stay untouched.

In [1]:
import re
from pathlib import Path

import pandas as pd
from supabase import create_client

DATA_DIR = Path("data")
DB_DIR = DATA_DIR / "database_files"


def load_env_local(path=".env.local"):
    env = {}
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, val = line.split("=", 1)
        env[key.strip()] = val.strip().strip('"').strip("'")
    return env


env = load_env_local()
SUPABASE_URL = env["NEXT_PUBLIC_SUPABASE_URL"]
SUPABASE_KEY = env["NEXT_PUBLIC_SUPABASE_PUBLISHABLE_KEY"]

client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("Connected to:", SUPABASE_URL)

Connected to: https://mjkwcrihtsxkujufzpfu.supabase.co


## Load CSVs + helpers

In [2]:
TABLES = {
    "prospects": "01_prospects.csv",
    "profiles": "02_prospect_profiles.csv",
    "sources": "03_prospect_sources.csv",
    "scores": "04_prospect_scores.csv",
    "analysis": "05_prospect_analysis.csv",
    "pipeline": "06_prospect_pipeline.csv",
    "sectors": "07_prospect_sectors.csv",
    "diligence": "08_prospect_diligence.csv",
}

dfs = {key: pd.read_csv(DB_DIR / fname) for key, fname in TABLES.items()}
for key, df in dfs.items():
    print(key, df.shape)


def to_records(df):
    # df.where(pd.notna(df), None) doesn't work reliably: pandas coerces None back to
    # NaN on float64 columns, which then serializes as an invalid JSON "NaN" literal.
    # Fix it up per-cell after to_dict instead.
    records = df.to_dict("records")
    for r in records:
        for k, v in r.items():
            if isinstance(v, float) and pd.isna(v):
                r[k] = None
    return records


def chunked(seq, size=200):
    for i in range(0, len(seq), size):
        yield seq[i : i + size]


def upsert_table(table_name, records, on_conflict):
    total = 0
    for batch in chunked(records):
        client.table(table_name).upsert(batch, on_conflict=on_conflict).execute()
        total += len(batch)
    print(f"upserted {total} rows into {table_name}")

prospects (149, 6)
profiles (149, 13)
sources (149, 7)
scores (149, 10)
analysis (149, 5)
pipeline (149, 6)
sectors (969, 2)
diligence (149, 4)


## 1. Push `prospects` (must go first — everything else FKs into it)

In [3]:
records = to_records(dfs["prospects"])
upsert_table("prospects", records, on_conflict="id")

upserted 149 rows into prospects


## 2. Push `prospect_profiles` (fill NOT NULL defaults for the payload only)

In [4]:
profiles_payload = dfs["profiles"].copy()

text_cols = ["family_or_group_background", "investment_philosophy"]
exposure_cols = [
    "infrastructure_exposure", "energy_exposure", "logistics_transport_exposure",
    "real_estate_exposure", "technology_digital_exposure", "emerging_markets_exposure",
]
bool_cols = [
    "long_horizon_capital_indicators", "permanent_capital_indicators",
    "direct_investment_activity", "governance_stewardship_language",
]

profiles_payload[text_cols] = profiles_payload[text_cols].fillna("")
profiles_payload[exposure_cols] = profiles_payload[exposure_cols].fillna("None")
profiles_payload[bool_cols] = profiles_payload[bool_cols].fillna(False)

records = to_records(profiles_payload)
upsert_table("prospect_profiles", records, on_conflict="prospect_id")

upserted 149 rows into prospect_profiles


C:\Users\USER\AppData\Local\Temp\ipykernel_18280\106608400.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  profiles_payload[bool_cols] = profiles_payload[bool_cols].fillna(False)


## 3. Push `prospect_sources`

In [5]:
sources_payload = dfs["sources"].copy()
sources_payload["source_quality"] = sources_payload["source_quality"].fillna("Medium")  # NOT NULL, default 'Medium'

records = to_records(sources_payload)
upsert_table("prospect_sources", records, on_conflict="prospect_id")

upserted 149 rows into prospect_sources


## 4. Push `prospect_scores` (priority overwritten from Master's raw tier labels)

In [6]:
scores_payload = dfs["scores"].copy()
scores_payload["classification"] = scores_payload["classification"].fillna("Not Suitable")  # NOT NULL, no db default
scores_payload["suitability_score"] = scores_payload["suitability_score"].fillna(0)
for col in ["family_office_fit", "permanent_capital_orientation", "sector_alignment",
            "governance_institutional_mindset", "strategic_adjacency_tbp", "engagement_readiness"]:
    scores_payload[col] = scores_payload[col].fillna(0)

records = to_records(scores_payload)
upsert_table("prospect_scores", records, on_conflict="prospect_id")

upserted 149 rows into prospect_scores


## 5. Push `prospect_analysis`

In [7]:
analysis_payload = dfs["analysis"].copy()
text_cols = ["tbp_relevance_summary", "best_tbp_entry_point", "suggested_conversation_angle", "recommended_contact_route"]
analysis_payload[text_cols] = analysis_payload[text_cols].fillna("")

records = to_records(analysis_payload)
upsert_table("prospect_analysis", records, on_conflict="prospect_id")

upserted 149 rows into prospect_analysis


## 6. Push `prospect_pipeline` (next_action_date has no db default — fill it)

In [8]:
pipeline_payload = dfs["pipeline"].copy()
pipeline_payload["pipeline_stage"] = pipeline_payload["pipeline_stage"].fillna("Identified")
pipeline_payload["assigned_owner"] = pipeline_payload["assigned_owner"].fillna("TBP Advisory")
pipeline_payload["next_action"] = pipeline_payload["next_action"].fillna("")
pipeline_payload["next_action_date"] = pipeline_payload["next_action_date"].fillna("2026-07-31")
pipeline_payload["briefing_pack_status"] = pipeline_payload["briefing_pack_status"].fillna("Not Generated")

records = to_records(pipeline_payload)
upsert_table("prospect_pipeline", records, on_conflict="prospect_id")

upserted 149 rows into prospect_pipeline


## 7. Push `prospect_sectors` (true mirror)

No per-pair upsert is needed here: since every prospect's full sector list is already known locally, mirroring is simplest as delete-all-for-these-prospects, then insert-all-from-local — same end state as diffing and deleting only the stale pairs, less code.

In [9]:
all_prospect_ids = dfs["prospects"]["id"].tolist()

deleted = 0
for batch in chunked(all_prospect_ids, 200):
    resp = client.table("prospect_sectors").delete().in_("prospect_id", batch).execute()
    deleted += len(resp.data)
print(f"deleted {deleted} existing prospect_sectors rows for {len(all_prospect_ids)} prospects")

sector_records = to_records(dfs["sectors"])
inserted = 0
for batch in chunked(sector_records, 200):
    client.table("prospect_sectors").insert(batch).execute()
    inserted += len(batch)
print(f"inserted {inserted} prospect_sectors rows")

deleted 369 existing prospect_sectors rows for 149 prospects
inserted 969 prospect_sectors rows


## 8. Push `prospect_diligence` (delete-then-insert — no natural unique key)

In [10]:
deleted = 0
for batch in chunked(all_prospect_ids, 200):
    resp = client.table("prospect_diligence").delete().in_("prospect_id", batch).execute()
    deleted += len(resp.data)
print(f"deleted {deleted} existing prospect_diligence rows")

diligence_payload = dfs["diligence"].copy()
diligence_payload["content"] = diligence_payload["content"].fillna("")  # NOT NULL, no db default
diligence_records = to_records(diligence_payload)

inserted = 0
for batch in chunked(diligence_records, 200):
    client.table("prospect_diligence").insert(batch).execute()
    inserted += len(batch)
print(f"inserted {inserted} prospect_diligence rows")

deleted 149 existing prospect_diligence rows
inserted 149 prospect_diligence rows


## Verify: row counts + spot checks against live Supabase

In [11]:
SUPA_TABLE_NAMES = {
    "prospects": "prospects",
    "profiles": "prospect_profiles",
    "sources": "prospect_sources",
    "scores": "prospect_scores",
    "analysis": "prospect_analysis",
    "pipeline": "prospect_pipeline",
    "sectors": "prospect_sectors",
    "diligence": "prospect_diligence",
}

for key, table_name in SUPA_TABLE_NAMES.items():
    resp = client.table(table_name).select("*", count="exact", head=True).execute()
    local_count = len(dfs[key])
    remote_count = resp.count
    status = "OK" if local_count == remote_count else "MISMATCH"
    print(f"{table_name}: local={local_count} remote={remote_count}  [{status}]")

print()
sample_id = dfs["prospects"][dfs["prospects"]["region"] == "Indonesia"]["id"].iloc[-1]
print("spot check:", sample_id)
print(client.table("prospects").select("*").eq("id", sample_id).execute().data)
print(client.table("prospect_scores").select("*").eq("prospect_id", sample_id).execute().data)
print(client.table("prospect_pipeline").select("*").eq("prospect_id", sample_id).execute().data)
print(client.table("prospect_sectors").select("*").eq("prospect_id", sample_id).execute().data)

prospects: local=149 remote=149  [OK]
prospect_profiles: local=149 remote=149  [OK]
prospect_sources: local=149 remote=149  [OK]
prospect_scores: local=149 remote=149  [OK]
prospect_analysis: local=149 remote=149  [OK]
prospect_pipeline: local=149 remote=149  [OK]
prospect_sectors: local=969 remote=969  [OK]
prospect_diligence: local=149 remote=149  [OK]

spot check: id-034
[{'id': 'id-034', 'prospect_name': 'PT Temas Tbk', 'prospect_type': 'Listed Corporate (IDX: TMAS)', 'country': 'Indonesia', 'city': 'Jakarta', 'region': 'Indonesia', 'created_at': '2026-07-05T08:13:26.020626+00:00', 'updated_at': '2026-07-05T08:13:26.020626+00:00'}]
[{'prospect_id': 'id-034', 'suitability_score': 83, 'classification': 'Priority Founding Steward Prospect', 'family_office_fit': 16, 'permanent_capital_orientation': 16, 'sector_alignment': 19, 'governance_institutional_mindset': 12, 'strategic_adjacency_tbp': 13, 'engagement_readiness': 7, 'scored_at': '2026-07-05T08:14:00.108469+00:00', 'priority': 'Hi